# Tutorial: Calculus & Optimization

> **Type:** Tutorial. Follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Module 03 (linear algebra, specifically the linear regression setup)  
> **What you'll build:** Gradient descent from scratch, shown on a parabola and then used to train a linear regression model. Plus a learning rate comparison and a scipy optimisation demo.

---

## What We're Building

Four sections:
1. Visualise a loss surface and implement gradient descent in 20 lines
2. Train linear regression with gradient descent (no sklearn)
3. Show what happens with learning rates that are too high, too low, and just right
4. Use `scipy.optimize.minimize`, and see why we can't use it for neural nets

## Step 0: Setup

In [ ]:
# !pip install numpy scipy matplotlib scikit-learn

import numpy as np
import scipy.optimize as opt
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['font.size'] = 11
print('Setup complete.')

---

## Part 1: Gradient Descent on a Parabola

### Step 1: The Loss Surface

Start simple: a 1D parabola f(x) = x². The minimum is at x=0. We know that, but let's pretend we don't and find it with gradient descent.

In [ ]:
# The function and its derivative
def f(x):
    return x**2

def df(x):
    return 2 * x  # derivative of x² is 2x

x_range = np.linspace(-4, 4, 300)

plt.figure(figsize=(8, 4))
plt.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x) = x² (loss surface)')
plt.xlabel('x (parameter value)')
plt.ylabel('f(x) (loss)')
plt.title('Loss Surface: f(x) = x²')
plt.legend()
plt.tight_layout()
plt.show()

# Check: what does the gradient tell us at x=2?
x_test = 2.0
print(f"At x = {x_test}: f(x) = {f(x_test)}, gradient = {df(x_test)}")
print(f"Gradient is positive → f is increasing → we should move LEFT (decrease x)")

**Chart description.** The curve f(x) = x² from x = −4 to 4. It's a U shape with its minimum at x = 0, where the loss is 0.

### Step 2: Implement Gradient Descent

In [ ]:
def gradient_descent_1d(start_x, learning_rate, n_steps):
    """Run gradient descent on f(x) = x². Returns history of (x, f(x)) pairs."""
    x = start_x
    history = [(x, f(x))]
    
    for step in range(n_steps):
        gradient = df(x)           # compute gradient
        x = x - learning_rate * gradient  # step opposite to gradient
        history.append((x, f(x)))
    
    return history

# Run it
history = gradient_descent_1d(start_x=3.0, learning_rate=0.2, n_steps=20)

# Print first few steps
print("Step | x value   | f(x) = loss")
print("-" * 40)
for i, (x, loss) in enumerate(history[:8]):
    print(f"  {i:2d} | {x:+.6f} | {loss:.6f}")
print(f"  ...")
print(f"  {len(history)-1:2d} | {history[-1][0]:+.6f} | {history[-1][1]:.8f}")
print(f"\nFinal x = {history[-1][0]:.8f}  (true minimum = 0)")

### Step 3: Visualise the Steps

In [ ]:
history_xs  = [h[0] for h in history]
history_fs  = [h[1] for h in history]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: steps on the loss surface
axes[0].plot(x_range, f(x_range), 'b-', linewidth=2, alpha=0.5, label='f(x) = x²')
axes[0].scatter(history_xs, history_fs, c=range(len(history)), cmap='YlOrRd', 
                s=60, zorder=5, label='GD steps')
axes[0].plot(history_xs, history_fs, 'k-', alpha=0.3, linewidth=1)
# Annotate first and last
axes[0].annotate(f'Start\nx={history_xs[0]:.1f}', (history_xs[0], history_fs[0]),
                 textcoords='offset points', xytext=(10,10), fontsize=9)
axes[0].annotate(f'End\nx≈{history_xs[-1]:.4f}', (history_xs[-1], history_fs[-1]),
                 textcoords='offset points', xytext=(5,-20), fontsize=9)
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].set_title('Gradient Descent Steps on Loss Surface')
axes[0].legend()

# Right: loss curve over steps
axes[1].plot(history_fs, 'o-', color='#E15759', markersize=5)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Loss f(x)')
axes[1].set_title('Loss Curve (should decrease monotonically)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig('gradient_descent_1d.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Left: gradient descent steps on f(x) = x², starting at x = 3 and ending near 0. The steps get shorter as they approach the minimum. Right: the loss at each of 20 steps on a log scale. It falls in a straight line from about 9 to below 10⁻⁸, so the loss shrinks by the same factor each step.

> **What just happened?** Gradient descent started at x=3 and took 20 steps to reach x≈0. Each step moved against the gradient. The loss curve (right) falls exponentially. That's what a healthy training curve looks like.

---

## Part 2: Linear Regression with Gradient Descent

Now apply gradient descent to an ML problem: train linear regression without sklearn.

### Step 4: Generate Data

In [ ]:
# Generate a simple regression dataset
X_raw, y = make_regression(n_samples=200, n_features=1, noise=20, random_state=RANDOM_SEED)

# Standardise features: CRITICAL for gradient descent
# (unnormalised features cause very different gradient magnitudes per parameter)
scaler = StandardScaler()
X_raw  = scaler.fit_transform(X_raw)

# Add bias column
X = np.column_stack([np.ones(len(X_raw)), X_raw])  # shape (200, 2)
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"y range: [{y.min():.1f}, {y.max():.1f}]")

### Step 5: The MSE Loss and Its Gradient

Loss: L(w) = (1/N) Σ (Xw - y)² = (1/N) ||Xw - y||²

Gradient: ∇L(w) = (2/N) Xᵀ(Xw - y)

In [ ]:
def mse_loss(w, X, y):
    residuals = X @ w - y
    return np.mean(residuals**2)

def mse_gradient(w, X, y):
    N = len(y)
    residuals = X @ w - y
    return (2 / N) * X.T @ residuals

# Verify gradient at a random starting point
w_test = np.array([0.5, -1.0])
print(f"Loss at test weights: {mse_loss(w_test, X, y):.4f}")
print(f"Gradient at test weights: {mse_gradient(w_test, X, y)}")
print("(Gradient direction tells us how to change w to reduce loss)")

### Step 6: Train with Gradient Descent

In [ ]:
def train_linear_regression_gd(X, y, learning_rate=0.1, n_epochs=200):
    """Train linear regression using gradient descent."""
    w = np.zeros(X.shape[1])  # initialise weights at zero
    loss_history = []
    
    for epoch in range(n_epochs):
        loss = mse_loss(w, X, y)
        grad = mse_gradient(w, X, y)
        w    = w - learning_rate * grad  # gradient descent step
        loss_history.append(loss)
        
        if epoch % 40 == 0:
            print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | w = [{w[0]:.4f}, {w[1]:.4f}]")
    
    return w, loss_history

w_gd, loss_history = train_linear_regression_gd(X, y, learning_rate=0.1, n_epochs=200)

### Step 7: Verify Against sklearn and the Normal Equations

In [ ]:
# Normal equations (exact solution)
w_exact = np.linalg.solve(X.T @ X, X.T @ y)

# sklearn
lr = LinearRegression(fit_intercept=False)  # bias already in X
lr.fit(X, y)
w_sklearn = lr.coef_

print("Weight comparison:")
print(f"  Gradient descent:  [{w_gd[0]:.6f}, {w_gd[1]:.6f}]")
print(f"  Normal equations:  [{w_exact[0]:.6f}, {w_exact[1]:.6f}]")
print(f"  sklearn:           [{w_sklearn[0]:.6f}, {w_sklearn[1]:.6f}]")
print(f"\nAll close: {np.allclose(w_gd, w_exact, atol=0.01)}")

### Step 8: Plot the Loss Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Loss curve
axes[0].plot(loss_history, color='#E15759', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training Loss Curve\n(decreasing → model is learning)')
axes[0].axhline(mse_loss(w_exact, X, y), color='grey', linestyle='--', 
                label='Optimal loss (normal equations)')
axes[0].legend()

# Fit vs data
axes[1].scatter(X_raw, y, alpha=0.5, s=30, color='#4E79A7', label='Data')
x_line = np.array([[1, X_raw.min()], [1, X_raw.max()]])
axes[1].plot([X_raw.min(), X_raw.max()], 
             [x_line[0] @ w_gd, x_line[1] @ w_gd],
             color='#E15759', linewidth=2, label='GD fit')
axes[1].set_xlabel('Feature (standardised)')
axes[1].set_ylabel('Target y')
axes[1].set_title('Learned Fit')
axes[1].legend()

plt.tight_layout()
plt.savefig('gd_linear_regression.png', dpi=150, bbox_inches='tight')
plt.show()

**Chart description.** Left: the training loss (MSE) over 200 epochs of gradient descent on a linear regression. It drops from about 7,000 to the optimal loss within about 15 epochs, then stays flat on the dashed line that marks the normal-equation solution. Right: the data with the fitted line. The line runs through the middle of the points.

---

## Part 3: Learning Rate Comparison

### Step 9: Too High, Too Low, Just Right

In [ ]:
learning_rates = {
    'Too small (0.001)':  0.001,
    'Just right (0.1)':   0.1,
    'Too large (1.5)':    1.5,
}

colours = {'Too small (0.001)': '#4E79A7', 'Just right (0.1)': '#59A14F', 'Too large (1.5)': '#E15759'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
optimal_loss = mse_loss(w_exact, X, y)

for label, lr_val in learning_rates.items():
    w = np.zeros(X.shape[1])
    history = []
    
    for epoch in range(100):
        loss = mse_loss(w, X, y)
        # Clip loss to avoid infinite values
        history.append(min(loss, 1e8))
        grad = mse_gradient(w, X, y)
        w = w - lr_val * grad
    
    axes[0].plot(history, label=label, color=colours[label], linewidth=2)
    axes[1].plot(history[:30], label=label, color=colours[label], 
                 linewidth=2, marker='o', markersize=4)

for ax in axes:
    ax.axhline(optimal_loss, color='grey', linestyle='--', alpha=0.7, label='Optimal loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend(fontsize=9)

axes[0].set_title('Loss Curves (100 epochs)')
axes[0].set_ylim(0, 5000)  # clip display of diverging run
axes[1].set_title('First 30 Epochs (zoom)')
axes[1].set_ylim(0, 5000)

plt.suptitle('Effect of Learning Rate on Training', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('learning_rate_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations:")
print("  Too small: converges, but slowly — needs many more epochs")
print("  Just right: within 1% of optimal by epoch ~17, converged by ~30")
print("  Too large: loss explodes (diverges) — gradient descent overshoots the minimum")

**Chart description.** Two charts of training loss for three learning rates. The left shows 100 epochs, the right the first 30. The y-axis is cut off at 5,000. With 0.1, the loss falls fast and reaches the optimal loss by about epoch 17. With 0.001, the loss stays near 5,000 and barely moves. With 1.5, the loss explodes, so its line doesn't appear on the chart.

---

## Part 4: scipy.optimize — When to Use It

### Step 10: scipy Solves Optimisation Problems Elegantly

In [ ]:
# scipy.optimize.minimize finds the minimum of any function
# It's excellent for low-dimensional, smooth, classical optimisation

def loss_fn(w):
    return mse_loss(w, X, y)

def gradient_fn(w):
    return mse_gradient(w, X, y)

# L-BFGS-B: a quasi-Newton method (uses gradient + curvature estimates)
result = opt.minimize(
    fun=loss_fn,
    x0=np.zeros(X.shape[1]),   # starting point
    jac=gradient_fn,            # gradient (optional but faster)
    method='L-BFGS-B'
)

print(f"scipy result:")
print(f"  Success:   {result.success}")
print(f"  Iterations: {result.nit}")
print(f"  Final loss: {result.fun:.6f}")
print(f"  Weights:    {result.x}")
print(f"\nGD weights: {w_gd}")
print(f"Normal eq:  {w_exact}")
print(f"\nscipy converges faster (fewer iterations) than our hand-rolled GD")

### Step 11: Why scipy Won't Work for Neural Networks

In [ ]:
# scipy works well for this 2-parameter problem (intercept + slope)
# But neural networks have millions of parameters

# GPT-3 has 175 billion parameters
# A small neural net might have 1 million parameters
# Newton's method stores the exact Hessian: an M×M matrix.
# L-BFGS-B ("limited-memory") keeps only ~10 pairs of recent
# gradient differences, so its memory grows linearly with M.

M = 1_000_000  # 1 million parameters
hessian_bytes = M * M * 4  # float32
lbfgs_bytes = 20 * M * 8   # 10 pairs of float64 vectors
print(f"Exact Hessian, 1M parameters: {hessian_bytes / 1e9:,.0f} GB "
      f"({hessian_bytes / 2**30:,.0f} GiB)")
print(f"L-BFGS-B history, 1M parameters: {lbfgs_bytes / 1e9:.2f} GB")
print("(An A100 GPU has 80 GB: the exact Hessian can't fit, L-BFGS can)")
print()
print("Why mini-batch SGD (not scipy) is used for neural networks:")
print("  1. L-BFGS-B needs exact full-batch gradients; mini-batch noise")
print("     breaks its line search and curvature estimates")
print("  2. Mini-batch noise helps SGD/Adam escape saddle points")
print("  3. Data doesn't fit in memory: must process in batches anyway")
print()
print("Rule of thumb:")
print("  Smooth loss, full data fits in memory -> scipy.optimize")
print("  Neural net, mini-batch training       -> Adam / SGD")


---

## What You Built

You implemented gradient descent from scratch on a parabola (20 lines). You extended it to train a linear regression model from scratch, matching sklearn and the normal equations exactly. You saw the effect of three learning rates, and where scipy fits compared with where mini-batch gradient descent is needed.

## What's Next

- Read `explanation.md` if you haven't. The chain rule and backpropagation section is the bridge to Module 28 (neural networks).
- Keep `reference.md` open. The gradient descent variants table and loss function formulas will come up again and again.
- **Extension challenge:** implement momentum on top of the basic gradient descent loop. Add a velocity variable: `v = beta * v + (1-beta) * grad`, then `w = w - lr * v`. Try `beta=0.9` and compare the loss curve.

**Next module:** `05_information_theory`: entropy, cross-entropy, and KL divergence, the mathematical language of probability-based loss functions.

---

**AWS SageMaker note:** SageMaker Training Jobs manage the compute for gradient descent at scale. You write the training loop, and SageMaker provisions the GPU instances. We'll submit a training job in Module 26. The gradient descent fundamentals here are exactly what runs inside it.